<a href="https://colab.research.google.com/github/XINRUIQI/CASA0025-Big-Data/blob/main/W05_quiz_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 Quiz

This notebook contains the SQL Quiz for Week 5. Section 1 uses the New York City data we know and love. Section 2 will conduct a spatial damage assessment across the Gaza strip.

### INSTRUCTIONS:

Run this notebook in Google Colab. The answer to each quesiton will be a number or a string. Input these into the corresponding question on Moodle. You have 90 minutes to attempt the quiz, so if you get stuck on a question, move on.

Make sure you run all of the cells of code in order. If you run into serious problems, try clicking on the "runtime" tab above and selecting "restart session and run all".

# Section 1



This URL: `https://s3.amazonaws.com/s3.cleverelephant.ca/postgis-workshop-2020.zip` points to a .zip file containing shapefiles associated with the NYC data that we've been using in class so far. You can choose how to use sql, either the `con.sql()` syntax or the `%%sql` syntax. Either way, use the code cell below to install and import the necessary packages.

In [ ]:
%pip install duckdb leafmap

In [ ]:
import duckdb
import leafmap

In [ ]:
url = "https://s3.amazonaws.com/s3.cleverelephant.ca/postgis-workshop-2020.zip"
!rm -rf postgis-workshop-2020.zip postgis-workshop-2020 nyc_data
leafmap.download_file(url, unzip=True, overwrite=True)

Downloading...
From: https://s3.amazonaws.com/s3.cleverelephant.ca/postgis-workshop-2020.zip
To: /content/postgis-workshop-2020.zip
100%|██████████| 22.5M/22.5M [00:00<00:00, 96.9MB/s]


Extracting files...


'/content/postgis-workshop-2020.zip'

In [ ]:
!ls
!find . -maxdepth 3 -name "nyc_*.shp"

postgis-workshop  postgis-workshop-2020.zip
./postgis-workshop/data/nyc_homicides.shp
./postgis-workshop/data/nyc_streets.shp
./postgis-workshop/data/nyc_subway_stations.shp
./postgis-workshop/data/nyc_census_blocks.shp
./postgis-workshop/data/nyc_neighborhoods.shp


In [ ]:
con = duckdb.connect("section1.duckdb")
con.install_extension("spatial")
con.load_extension("spatial")

con.sql("CREATE SCHEMA IF NOT EXISTS s1;")
con.sql("SET schema='s1';")

In [ ]:
!ls

postgis-workshop	   section1.duckdb
postgis-workshop-2020.zip  section1.duckdb.wal


## Question 1

Firslty, please provide the link to your colab notebook in the Moodle quiz using the following steps:

1. In the top right corner, click the "share" button
2. In the popup, click on "restricted" and change this to "anyone with the link"
3. Finally, click "copy link", and paste the link into the box for Question 1 on moodle.

Once you've done this, download and unzip the data, then create the following tables using the corresponding shapefiles.

- nyc_neighborhoods
- nyc_census_blocks
- nyc_homicides
- nyc_streets
- nyc_subway_stations

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE nyc_neighborhoods AS
SELECT * FROM ST_Read('postgis-workshop/data/nyc_neighborhoods.shp');
""")

con.sql("""
CREATE OR REPLACE TABLE nyc_census_blocks AS
SELECT * FROM ST_Read('postgis-workshop/data/nyc_census_blocks.shp');
""")

con.sql("""
CREATE OR REPLACE TABLE nyc_homicides AS
SELECT * FROM ST_Read('postgis-workshop/data/nyc_homicides.shp');
""")

con.sql("""
CREATE OR REPLACE TABLE nyc_streets AS
SELECT * FROM ST_Read('postgis-workshop/data/nyc_streets.shp');
""")

con.sql("""
CREATE OR REPLACE TABLE nyc_subway_stations AS
SELECT * FROM ST_Read('postgis-workshop/data/nyc_subway_stations.shp');
""")

In [ ]:
con.sql("SHOW TABLES FROM s1;").df()

,name
0,nyc_census_blocks
1,nyc_homicides
2,nyc_neighborhoods
3,nyc_streets
4,nyc_subway_stations


In [ ]:
con.sql("""
SELECT
  (SELECT COUNT(*) FROM nyc_neighborhoods) AS n_neigh,
  (SELECT COUNT(*) FROM nyc_census_blocks) AS n_blocks,
  (SELECT COUNT(*) FROM nyc_homicides) AS n_homicides,
  (SELECT COUNT(*) FROM nyc_streets) AS n_streets,
  (SELECT COUNT(*) FROM nyc_subway_stations) AS n_stations
""").df()

,n_neigh,n_blocks,n_homicides,n_streets,n_stations
0,129,38794,3984,19091,491


## Question 2:
What is the longest street in New York? Note, it may be split up into multiple segments! Ignore missing and null values.


In [ ]:
con.sql("""
SELECT
  name,
  ST_AsText(ST_StartPoint(geom)) AS start_pt
FROM nyc_streets
WHERE geom IS NOT NULL
LIMIT 10;
""").df()

,NAME,start_pt
0,Shore Pky S,POINT (586785.4767897038 4492901.0014554765)
1,None,POINT (586645.0073625665 4504977.750360583)
2,Avenue O,POINT (586750.3019977848 4496109.72213903)
3,Walsh Ct,POINT (586728.695515043 4497971.05313857)
4,None,POINT (586587.0531467082 4510088.250402982)
5,Avenue Z,POINT (586792.1590947693 4493279.321965924)
6,Dank Ct,POINT (586794.7541421958 4493361.728529104)
7,Cumberland Walk,POINT (586657.467661773 4505324.904212245)
8,Cumberland Walk,POINT (586670.7115426222 4505521.566761277)
9,None,POINT (586598.3257999844 4510424.446496345)


In [ ]:
con.sql("""
SELECT
  name,
  SUM(ST_Length(geom)) AS total_len_m
FROM nyc_streets
WHERE name IS NOT NULL AND TRIM(name) <> ''
GROUP BY name
ORDER BY total_len_m DESC
LIMIT 1;
""").df()

,NAME,total_len_m
0,Grand Central Pky,40210.484038


## Question 3:

Which borough had the fewest daytime shootings in 2009?

In [ ]:
con.sql("""
SELECT *
FROM nyc_homicides
LIMIT 5;
""").df()

,INCIDENT_D,BORONAME,NUM_VICTIM,PRIMARY_MO,ID,WEAPON,LIGHT_DARK,YEAR,geom
0,2008-01-01,Brooklyn,1,None,7,gun,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,2008-01-04,Manhattan,1,None,14,gun,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,2008-01-05,Queens,1,None,15,gun,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,2008-01-04,Queens,1,None,16,knife,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,2008-01-05,Queens,1,None,18,gun,D,2008,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [ ]:
con.sql("""
SELECT
  LIGHT_DARK,
  COUNT(*)
FROM nyc_homicides
GROUP BY LIGHT_DARK
""").df()

,LIGHT_DARK,count_star()
0,None,2168
1,D,1211
2,L,605


In [ ]:
con.sql("""
SELECT
  BORONAME AS boro,
  COUNT(*) AS n
FROM nyc_homicides
WHERE YEAR = 2009
  AND LIGHT_DARK = 'L'
  AND WEAPON = 'gun'
GROUP BY BORONAME
ORDER BY n ASC
LIMIT 1;
""").df()

,boro,n
0,Staten Island,2


## Question 4:

What is the total population of the census blocks served by the L train?

In [ ]:
df_info = con.sql("PRAGMA table_info('nyc_subway_stations');").df()
df_info

,cid,name,type,notnull,dflt_value,pk
0,0,OBJECTID,DOUBLE,False,None,False
1,1,ID,DOUBLE,False,None,False
2,2,NAME,VARCHAR,False,None,False
3,3,ALT_NAME,VARCHAR,False,None,False
4,4,CROSS_ST,VARCHAR,False,None,False
5,5,LONG_NAME,VARCHAR,False,None,False
6,6,LABEL,VARCHAR,False,None,False
7,7,BOROUGH,VARCHAR,False,None,False
8,8,NGHBHD,VARCHAR,False,None,False
9,9,ROUTES,VARCHAR,False,None,False


In [ ]:
con.sql("SELECT * FROM nyc_subway_stations LIMIT 5;").df()

,OBJECTID,ID,NAME,ALT_NAME,CROSS_ST,LONG_NAME,LABEL,BOROUGH,NGHBHD,ROUTES,TRANSFERS,COLOR,EXPRESS,CLOSED,geom
0,1.0,376.0,Cortlandt St,None,Church St,"Cortlandt St (R,W) Manhattan","Cortlandt St (R,W)",Manhattan,None,"R,W","R,W",YELLOW,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,2.0,2.0,Rector St,None,None,Rector St (1) Manhattan,Rector St (1),Manhattan,None,1,1,RED,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,3.0,1.0,South Ferry,None,None,South Ferry (1) Manhattan,South Ferry (1),Manhattan,None,1,1,RED,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,4.0,125.0,138th St,Grand Concourse,Grand Concourse,"138th St / Grand Concourse (4,5) Bronx","138th St / Grand Concourse (4,5)",Bronx,None,"4,5","4,5",GREEN,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,5.0,126.0,149th St,Grand Concourse,Grand Concourse,149th St / Grand Concourse (4) Bronx,149th St / Grand Concourse (4),Bronx,None,4,"2,4,5",GREEN,express,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [ ]:
df_total_pop = con.sql("""
WITH l_stations AS (
  SELECT geom
  FROM nyc_subway_stations
  WHERE ROUTES LIKE '%L%'
),
served_blocks AS (
  SELECT DISTINCT cb."POPN_TOTAL"
  FROM nyc_census_blocks cb
  JOIN l_stations s
    ON ST_Contains(cb.geom, s.geom)
)
SELECT COALESCE(SUM("POPN_TOTAL"), 0) AS total_pop
FROM served_blocks;
""").df()

df_total_pop

,total_pop
0,7147.0


In [ ]:
con.sql("""
SELECT COUNT(*) AS n_l_stations
FROM nyc_subway_stations
WHERE ROUTES LIKE '%L%';
""").df()

,n_l_stations
0,24


## Question 5

which subway station had the largest number of homicides within 1km?

In [ ]:
con.sql("""
SELECT
  s.name AS station_name,
  COUNT(*) AS homicide_count
FROM nyc_subway_stations s
JOIN nyc_homicides h
  ON ST_DWithin(s.geom, h.geom, 1000)
GROUP BY s.name
ORDER BY homicide_count DESC
LIMIT 1;
""").df()

,station_name,homicide_count
0,125th St,305


In [ ]:
con.sql("""
SELECT
  s.name AS station_name,
  COUNT(*) AS homicide_count
FROM nyc_subway_stations s
JOIN nyc_homicides h
  ON ST_DWithin(s.geom, h.geom, 1000)
GROUP BY s.name
ORDER BY homicide_count DESC
LIMIT 10;
""").df()

,station_name,homicide_count
0,125th St,305
1,116th St,261
2,Franklin Ave,251
3,145th St,246
4,170th St,214
5,Utica Ave,213
6,135th St,208
7,Sutter Ave,208
8,Myrtle Ave,206
9,Broadway Jct,204


# Section 2

This section explores building damage in the Gaza Strip resulting from the ongoing war. You will conduct a geospatial analysis of building damage using two datasets:

1. [Humantiarian Open Street Map](https://www.hotosm.org/projects/gaza-building-footprints-pre-conflict-update-2024/) building footprints for the Gaza Strip
  * data: https://storage.googleapis.com/qm2/Gaza_Buildings_2.geojson.zip
2. [Damage points](https://unosat.org/products/3984) from the UN Satellite Agency (UNOSAT).
  * data: https://storage.googleapis.com/qm2/UNOSAT_GAZA_20240503_2.zip


The Coordinate Reference System of these datasets is EPSG:4326; therefore, stock functions like ST_DISTANCE() will yield values in degrees, not meters. Be mindful of this in your analysis, and use the DuckDB spatial functions [documentation](https://duckdb.org/docs/extensions/spatial/functions#st_geomfromtext) to your advantage.


First, download and unzip the data, and create two tables: `gaza_buildings`, containing the building footprint data, and `damage_points` containing the UNOSAT damage points. Make sure to set a spatial index on these tables, this will make your queries run much faster!

In [ ]:
buildings_url = "https://storage.googleapis.com/qm2/Gaza_Buildings_2.geojson.zip"
damage_url    = "https://storage.googleapis.com/qm2/UNOSAT_GAZA_20240503_2.zip"

!rm -rf Gaza_Buildings_2.geojson.zip UNOSAT_GAZA_20240503_2.zip

leafmap.download_file(buildings_url, unzip=True, overwrite=True)
leafmap.download_file(damage_url, unzip=True, overwrite=True)

Downloading...
From: https://storage.googleapis.com/qm2/Gaza_Buildings_2.geojson.zip
To: /content/Gaza_Buildings_2.geojson.zip
100%|██████████| 15.0M/15.0M [00:00<00:00, 120MB/s]


Extracting files...


Downloading...
From: https://storage.googleapis.com/qm2/UNOSAT_GAZA_20240503_2.zip
To: /content/UNOSAT_GAZA_20240503_2.zip
100%|██████████| 4.98M/4.98M [00:00<00:00, 76.0MB/s]


Extracting files...


'/content/UNOSAT_GAZA_20240503_2.zip'

In [ ]:
!ls -lah
!find . -maxdepth 3 -type f -name "*Gaza*" -o -name "*UNOSAT*"

total 391M
drwxr-xr-x 1 root root 4.0K Feb 10 16:22 .
drwxr-xr-x 1 root root 4.0K Feb 10 09:05 ..
drwxr-xr-x 4 root root 4.0K Jan 16 14:24 .config
-rw-r--r-- 1 root root 109M Feb 10 16:22 Gaza_Buildings_2.geojson
-rw-r--r-- 1 root root  15M Feb 10 16:22 Gaza_Buildings_2.geojson.zip
drwxr-xr-x 4 root root 4.0K Feb 10 09:06 postgis-workshop
-rw-r--r-- 1 root root  22M Feb 10 16:22 postgis-workshop-2020.zip
-rw-r--r-- 1 root root  12K Feb 10 16:22 section1.duckdb
-rw-r--r-- 1 root root  16M Feb 10 16:22 section1.duckdb.wal
-rw-r--r-- 1 root root    5 Feb 10 16:22 UNOSAT_GAZA_20240503_2.cpg
-rw-r--r-- 1 root root 220M Feb 10 16:22 UNOSAT_GAZA_20240503_2.dbf
-rw-r--r-- 1 root root  401 Feb 10 16:22 UNOSAT_GAZA_20240503_2.prj
-rw-r--r-- 1 root root 4.7M Feb 10 16:22 UNOSAT_GAZA_20240503_2.shp
-rw-r--r-- 1 root root 1.1M Feb 10 16:22 UNOSAT_GAZA_20240503_2.shx
-rw-r--r-- 1 root root 4.8M Feb 10 16:22 UNOSAT_GAZA_20240503_2.zip
./UNOSAT_GAZA_20240503_2.zip
./UNOSAT_GAZA_20240503_2.dbf
./UNOSAT

In [ ]:
con = duckdb.connect("section1.duckdb")

con.install_extension("spatial")
con.load_extension("spatial")

con.sql("CREATE SCHEMA IF NOT EXISTS s2;")
con.sql("SET schema='s2';")

## Question 6

Create a table called "gaza_buildings_damaged" which is comprised only of building footprints that intersect with a UNOSAT damage point. How many damaged buildings are there in Gaza?

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE gaza_buildings AS
SELECT * FROM ST_Read('Gaza_Buildings_2.geojson');
""")
con.sql("""
CREATE OR REPLACE TABLE damage_points AS
SELECT * FROM ST_Read('UNOSAT_GAZA_20240503_2.shp');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.sql("SHOW TABLES FROM s2;").df()

,name
0,damage_points
1,gaza_buildings


In [ ]:
con.sql("SELECT COUNT(*) AS n_buildings FROM gaza_buildings;").df()
con.sql("SELECT COUNT(*) AS n_points FROM damage_points;").df()

,n_points
0,135295


In [ ]:
con.sql("CREATE INDEX gaza_buildings_rtree ON gaza_buildings USING RTREE(geom);")
con.sql("CREATE INDEX damage_points_rtree ON damage_points USING RTREE(geom);")

con.sql("SELECT * FROM duckdb_indexes();").df()

,database_name,database_oid,schema_name,schema_oid,index_name,index_oid,table_name,table_oid,comment,tags,is_unique,is_primary,expressions,sql
0,section1,592,s2,2443,damage_points_rtree,2467,damage_points,2451,None,{},False,False,[geom],CREATE INDEX damage_points_rtree ON s2.damage_...
1,section1,592,s2,2443,gaza_buildings_rtree,2457,gaza_buildings,2445,None,{},False,False,[geom],CREATE INDEX gaza_buildings_rtree ON s2.gaza_b...


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE gaza_buildings_damaged AS
SELECT DISTINCT b.*
FROM gaza_buildings b
JOIN damage_points p
  ON ST_Intersects(b.geom, p.geom);
""")

In [ ]:
con.sql("""
SELECT COUNT(*) AS damaged_buildings
FROM gaza_buildings_damaged;
""").df()

,damaged_buildings
0,103147


## Question 7

What is the total area (in square kilometers) of damaged buildings in Gaza? note: there are TWO area functions in duckDB, and only one of them returns answers in meters.

In [ ]:
area_df = con.sql("""
SELECT
  SUM(ST_Area_Spheroid(geom)) / 1000000 AS area_km2
FROM gaza_buildings_damaged;
""").df()

area_df

,area_km2
0,19.578145


## Question 8

What percentage of hospitals in Gaza have been damaged?


In [ ]:
con.sql("DESCRIBE gaza_buildings;").df()

,column_name,column_type,null,key,default,extra
0,osm_id,INTEGER,YES,None,None,None
1,osm_type,VARCHAR,YES,None,None,None
2,building,VARCHAR,YES,None,None,None
3,name,VARCHAR,YES,None,None,None
4,geom,GEOMETRY,YES,None,None,None


In [ ]:
con.sql("""
SELECT building
FROM gaza_buildings
WHERE building IS NOT NULL
  AND lower(building) LIKE '%hosp%'
GROUP BY building;
""").df()

,building
0,hospital
1,yes;hospital


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE hospitals AS
SELECT *
FROM gaza_buildings
WHERE building IS NOT NULL
  AND lower(building) LIKE '%hospital%';
""")

con.sql("SELECT COUNT(*) AS total_hospitals FROM hospitals;").df()

,total_hospitals
0,53


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE hospitals_damaged AS
SELECT DISTINCT h.*
FROM hospitals h
JOIN gaza_buildings_damaged d
  ON h.osm_id = d.osm_id;
""")

con.sql("SELECT COUNT(*) AS damaged_hospitals FROM hospitals_damaged;").df()

,damaged_hospitals
0,29


In [ ]:
con.sql("""
SELECT
  100.0 * (SELECT COUNT(*) FROM hospitals_damaged)
       / NULLIF((SELECT COUNT(*) FROM hospitals), 0) AS pct_hospitals_damaged;
""").df()

,pct_hospitals_damaged
0,54.716981


## Question 9

The UNOSAT data contains a column called "Governorat" (shapefiles only allow column names of up to 10 characters). Calculate the number of damaged schools in each governorate.

In [ ]:
con.sql("""
SELECT * EXCLUDE geom
FROM damage_points
LIMIT 5;
""").df()

,OBJECTID,SiteID,SensorDate,SensorID,Confidence,Main_Damag,SensorDa_1,SensorID_2,Confiden_1,Main_Dam_1,...,Main_Dam_6,Damage_S_5,Grouped_Da,FieldValid,Notes,Territory,Governorat,Municipali,Neighborho,EventCode
0,1.0,26,None,<NA>,<NA>,<NA>,2023/11/07 00:00:00.000,8,1,1,...,1,0,1,0,None,Gaza Strip,North Gaza,Jabalya,Az Zohour,CE20231007PSE
1,2.0,26,None,<NA>,<NA>,<NA>,2023/11/07 00:00:00.000,8,1,1,...,1,0,1,0,None,Gaza Strip,North Gaza,Jabalya,Az Zohour,CE20231007PSE
2,3.0,26,None,<NA>,<NA>,<NA>,2023/11/07 00:00:00.000,8,2,3,...,3,0,1,0,None,Gaza Strip,North Gaza,Jabalya,Az Zohour,CE20231007PSE
3,4.0,26,None,<NA>,<NA>,<NA>,2023/11/07 00:00:00.000,8,2,3,...,2,0,1,0,None,Gaza Strip,North Gaza,Jabalya,Az Zohour,CE20231007PSE
4,5.0,26,None,<NA>,<NA>,<NA>,2023/11/07 00:00:00.000,8,2,3,...,2,0,1,0,None,Gaza Strip,North Gaza,Jabalya,Az Zohour,CE20231007PSE


In [ ]:
con.sql("""
SELECT DISTINCT Governorat
FROM damage_points
WHERE Governorat IS NOT NULL
ORDER BY Governorat;
""").df()

,Governorat
0,Deir Al-Balah
1,Gaza
2,Khan Yunis
3,North Gaza
4,Rafah


In [ ]:
con.sql("""
SELECT Grouped_Da, COUNT(*) AS n
FROM damage_points
WHERE Grouped_Da IS NOT NULL
GROUP BY Grouped_Da
ORDER BY n DESC;
""").df()

,Grouped_Da,n
0,1,135295


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE schools AS
SELECT *
FROM gaza_buildings
WHERE building IS NOT NULL
  AND lower(building) LIKE '%school%';
""")
con.sql("CREATE INDEX IF NOT EXISTS schools_geom_idx ON schools USING RTREE (geom);")
con.sql("CREATE INDEX IF NOT EXISTS damage_points_geom_idx ON damage_points USING RTREE (geom);")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE schools_damaged_by_gov AS
SELECT
  d.Governorat,
  s.osm_id
FROM schools s
JOIN damage_points d
  ON ST_Intersects(s.geom, d.geom)
WHERE d.Governorat IS NOT NULL;
""")

In [ ]:
con.sql("""
SELECT
  Governorat,
  COUNT(DISTINCT osm_id) AS damaged_schools
FROM schools_damaged_by_gov
GROUP BY Governorat
ORDER BY damaged_schools DESC;
""").df()


,Governorat,damaged_schools
0,Gaza,57
1,Khan Yunis,44
2,North Gaza,26
3,Deir Al-Balah,6


## Question 10

The "Dar Al-Shifa Hospital" was the largest medical complex in the gaza strip. Find the hospital's building footprint, and calculate the proportion of damaged buildings within 1 kilometer.

In [ ]:
con.sql("""
SELECT osm_id, name, building
FROM gaza_buildings
WHERE name IS NOT NULL
  AND lower(name) LIKE '%shifa%'
LIMIT 50;
""").df()

,osm_id,name,building
0,295746100,Dar Al-Shifa Hospital,yes
1,296110768,Al-Shifa Maternity and NICU,hospital
2,295746098,Al-Shifa Outpatient Clinic,hospital
3,296108369,Al-Shifa Hospital Pharmacy and Burns Unit,yes
4,296110754,Al-Shifa Pathology and Morgue,yes
5,295746120,Al-Shifa Internal Medicine and Dialysis,hospital
6,41305016,Al-Shifa Pharmacy,yes
7,296110604,Al-Shifa MRI Department,hospital
8,1224488858,Al-Shifa Obstetrics Hospital,hospital
9,295746102,Dar Al-Shifa Hospital Gate,yes


In [ ]:
con.sql("DESCRIBE gaza_buildings_damaged;").df()

,column_name,column_type,null,key,default,extra
0,osm_id,INTEGER,YES,None,None,None
1,osm_type,VARCHAR,YES,None,None,None
2,building,VARCHAR,YES,None,None,None
3,name,VARCHAR,YES,None,None,None
4,geom,GEOMETRY,YES,None,None,None


In [ ]:
con.sql("""
CREATE OR REPLACE TEMP TABLE dar_al_shifa AS
SELECT *
FROM gaza_buildings
WHERE name IS NOT NULL
  AND lower(name) LIKE '%shifa%'
  AND lower(name) NOT LIKE '%gate%'
ORDER BY
  CASE WHEN lower(name) = 'dar al-shifa hospital' THEN 0 ELSE 1 END,
  CASE WHEN lower(name) LIKE '%hospital%' THEN 0 ELSE 1 END,
  ST_Area(ST_Transform(geom, 'EPSG:4326', 'EPSG:32636'))
LIMIT 1;
""")

In [ ]:
con.sql("""
SELECT
  osm_id, name, building,
  ST_Area(ST_Transform(geom, 'EPSG:4326', 'EPSG:32636')) AS area_m2
FROM dar_al_shifa;
""").df()

,osm_id,name,building,area_m2
0,295746100,Dar Al-Shifa Hospital,yes,3999.935938


In [ ]:
con.sql("""
CREATE OR REPLACE TEMP TABLE buildings_1km AS
SELECT DISTINCT b.osm_id
FROM gaza_buildings b
JOIN dar_al_shifa h
  ON ST_DWithin(
       ST_Transform(b.geom, 'EPSG:4326', 'EPSG:32636'),
       ST_Transform(h.geom, 'EPSG:4326', 'EPSG:32636'),
       1000
     )
WHERE b.osm_id <> (SELECT osm_id FROM dar_al_shifa);
""")

In [ ]:
con.sql("SELECT COUNT(*) AS n_buildings_1km FROM buildings_1km;").df()

,n_buildings_1km
0,6316


In [ ]:
con.sql("""
WITH total AS (
  SELECT COUNT(osm_id) AS n_total
  FROM buildings_1km
),
damaged AS (
  SELECT COUNT(DISTINCT b1.osm_id) AS n_damaged
  FROM buildings_1km b1
  JOIN gaza_buildings_damaged d
    ON b1.osm_id = d.osm_id
)
SELECT
  n_damaged,
  n_total,
  (1.0 * n_damaged / NULLIF(n_total, 0)) AS prop_damaged,
  (100.0 * n_damaged / NULLIF(n_total, 0)) AS pct_damaged
FROM total, damaged;
""").df()

,n_damaged,n_total,prop_damaged,pct_damaged
0,2849,6316,0.451077,45.107663
